## Dati da inserire in tesi

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


DATA_DIRECTORY = Path("../results/brt")

PAIRS = [
    ("euclidean", "dce"),
    ("btn", "stn"),
    ("btn", "long_lat"),
    ("stn", "long_lat"),
    ("ttc", "time_eggert"),
]

LABELS = {
    "euclidean": "Euclidean",
    "dce": "DCE",
    "btn": "BTN",
    "stn": "STN",
    "long_lat": "min(BTN,STN)",
    "ttc": "TTC",
    "time_eggert": "Time Eggert",
}

STATE_NAMES = (
    "x_rel", "y_rel", "theta_rel", "v_H", "delta_E", "v_E",
)

rows = []

for first, second in PAIRS:
    with (
        np.load(DATA_DIRECTORY / f"{first}.npz", allow_pickle=False) as data_a,
        np.load(DATA_DIRECTORY / f"{second}.npz", allow_pickle=False) as data_b,
    ):
        if not all(
            np.array_equal(data_a[name], data_b[name])
            for name in STATE_NAMES
        ):
            raise ValueError(f"{first}/{second}: different grids.")

        if not np.isclose(
            float(data_a["target_time"]),
            float(data_b["target_time"]),
            rtol=0,
            atol=1e-7,
        ):
            raise ValueError(f"{first}/{second}: different BRT horizons.")

        for field in ("V0", "BRT"):
            a, b = data_a[field], data_b[field]

            if a.shape != b.shape:
                raise ValueError(f"{first}/{second}: incompatible shapes.")

            if not np.isfinite(a).all() or not np.isfinite(b).all():
                raise ValueError(f"{first}/{second}: non-finite values.")

            rows.append({
                "First": LABELS[first],
                "Second": LABELS[second],
                "Field": field,
                "First < Second (%)": 100 * np.count_nonzero(a < b) / a.size,
                "First <= Second (%)": 100 * np.count_nonzero(a <= b) / a.size,
            })

            del a, b

results = pd.DataFrame(rows)
display(results.style.format({
    "First < Second (%)": "{:.2f}",
    "First <= Second (%)": "{:.2f}",
}).hide(axis="index"))

First,Second,Field,First < Second (%),First <= Second (%)
Euclidean,DCE,V0,0.00,44.40
Euclidean,DCE,BRT,21.09,61.77
BTN,STN,V0,22.78,64.71
BTN,STN,BRT,27.57,66.76
BTN,"min(BTN,STN)",V0,0.00,64.71
BTN,"min(BTN,STN)",BRT,12.74,62.00
STN,"min(BTN,STN)",V0,0.00,77.22
STN,"min(BTN,STN)",BRT,13.88,63.36
TTC,Time Eggert,V0,0.03,74.21
TTC,Time Eggert,BRT,3.84,39.84


In [3]:
import numpy as np
import pandas as pd
from IPython.display import display

rows = []

for first, second in PAIRS:
    with (
        np.load(DATA_DIRECTORY / f"{first}.npz", allow_pickle=False) as data_a,
        np.load(DATA_DIRECTORY / f"{second}.npz", allow_pickle=False) as data_b,
    ):
        for field in ("V0", "BRT"):
            a = data_a[field]
            b = data_b[field]

            if a.shape != b.shape:
                raise ValueError(f"{first}/{second}: incompatible shapes.")

            # Accumulazione per blocchi per limitare l'uso di memoria
            total_difference = 0.0
            max_difference = -np.inf
            count = 0

            for k in range(a.shape[0]):
                mask = a[k] < b[k]

                differences = (
                    b[k][mask].astype(np.float64)
                    - a[k][mask].astype(np.float64)
                )

                if differences.size > 0:
                    total_difference += np.sum(differences)
                    max_difference = max(
                        max_difference,
                        np.max(differences)
                    )
                    count += differences.size

            mean_difference = (
                total_difference / count if count > 0 else np.nan
            )

            max_difference = (
                max_difference if count > 0 else np.nan
            )

            rows.append({
                "First": LABELS[first],
                "Second": LABELS[second],
                "Field": field,
                "First < Second (%)": 100 * count / a.size,
                "Mean Difference": mean_difference,
                "Max Difference": max_difference,
            })

results_difference = pd.DataFrame(rows)

display(results_difference.style.format({
    "First < Second (%)": "{:.2f}",
    "Mean Difference": "{:.4f}",
    "Max Difference": "{:.4f}",
}).hide(axis="index"))

First,Second,Field,First < Second (%),Mean Difference,Max Difference
Euclidean,DCE,V0,0.00,nan,nan
Euclidean,DCE,BRT,21.09,0.0470,3.7369
BTN,STN,V0,22.78,0.9865,12.6461
BTN,STN,BRT,27.57,0.9036,13.1880
BTN,"min(BTN,STN)",V0,0.00,nan,nan
BTN,"min(BTN,STN)",BRT,12.74,0.0489,7.8557
STN,"min(BTN,STN)",V0,0.00,nan,nan
STN,"min(BTN,STN)",BRT,13.88,0.0200,4.6290
TTC,Time Eggert,V0,0.03,1.3945,5.4733
TTC,Time Eggert,BRT,3.84,0.1454,8.4039


In [10]:
from pathlib import Path
import numpy as np

DATA_DIRECTORY = Path("../results/brt")

STATE_NAMES = (
    "x_rel", "y_rel", "theta_rel", "v_H", "delta_E", "v_E",
)
UNITS = ("m", "m", "rad", "m/s", "rad", "m/s")

with (
    np.load(DATA_DIRECTORY / "euclidean.npz", allow_pickle=False) as euclidean,
    np.load(DATA_DIRECTORY / "dce.npz", allow_pickle=False) as dce,
):
    if not all(
        np.array_equal(euclidean[name], dce[name])
        for name in STATE_NAMES
    ):
        raise ValueError("The grids do not match.")

    if not np.isclose(
        float(euclidean["target_time"]),
        float(dce["target_time"]),
        rtol=0,
        atol=1e-7,
    ):
        raise ValueError("The BRT horizons do not match.")

    v_euclidean = euclidean["BRT"]
    v_dce = dce["BRT"]

    if v_euclidean.shape != v_dce.shape:
        raise ValueError("The array shapes do not match.")

    if not np.isfinite(v_euclidean).all() or not np.isfinite(v_dce).all():
        raise ValueError("Non-finite values detected.")

    # Search by slabs to limit memory use.
    maximum_difference = -np.inf
    index = None

    for k in range(v_dce.shape[0]):
        difference = (
            v_dce[k].astype(np.float64)
            - v_euclidean[k].astype(np.float64)
        )
        local_index = np.unravel_index(
            np.argmax(difference), difference.shape
        )
        local_maximum = float(difference[local_index])

        if local_maximum > maximum_difference:
            maximum_difference = local_maximum
            index = (k, *local_index)

    if maximum_difference <= 0:
        print("No grid node satisfies V_DCE > V_Euclidean.")
    else:
        print("State with the largest difference V_DCE - V_Euclidean:\n")

        for name, unit, i in zip(STATE_NAMES, UNITS, index):
            axis = euclidean[name]
            boundary = "  [grid boundary]" if i in (0, len(axis) - 1) else ""
            # Relative heading is periodic, so its endpoints are not boundaries.
            if name == "theta_rel":
                boundary = ""
            print(f"{name:10s} = {float(axis[i]): .8g} {unit}{boundary}")

        print(f"\nGrid index: {index}")
        print(f"Target time: {float(euclidean['target_time']):g} s")
        print(f"V_Euclidean = {float(v_euclidean[index]):.10g}")
        print(f"V_DCE       = {float(v_dce[index]):.10g}")
        print(f"Difference  = {maximum_difference:.10g}")

del v_euclidean, v_dce

State with the largest difference V_DCE - V_Euclidean:

x_rel      =  5.0000005 m
y_rel      =  0 m
theta_rel  = -2.3561947 rad
v_H        =  1 m/s  [grid boundary]
delta_E    = -0.20943952 rad
v_E        =  1 m/s  [grid boundary]

Grid index: (17, np.int64(8), np.int64(6), np.int64(0), np.int64(1), np.int64(0))
Target time: -3 s
V_Euclidean = -13.16239262
V_DCE       = -9.425515175
Difference  = 3.736877441


In [12]:
from pathlib import Path
import numpy as np

DATA_DIRECTORY = Path("../results/brt")

STATE_NAMES = (
    "x_rel", "y_rel", "theta_rel", "v_H", "delta_E", "v_E",
)
UNITS = ("m", "m", "rad", "m/s", "rad", "m/s")

with (
    np.load(DATA_DIRECTORY / "euclidean.npz", allow_pickle=False) as euclidean,
    np.load(DATA_DIRECTORY / "dce.npz", allow_pickle=False) as dce,
):
    axes = tuple(euclidean[name] for name in STATE_NAMES)

    if not all(
        np.array_equal(axis, dce[name])
        for name, axis in zip(STATE_NAMES, axes)
    ):
        raise ValueError("The grids do not match.")

    if not np.isclose(
        float(euclidean["target_time"]),
        float(dce["target_time"]),
        rtol=0,
        atol=1e-7,
    ):
        raise ValueError("The BRT horizons do not match.")

    v_euclidean = euclidean["BRT"]
    v_dce = dce["BRT"]

    expected_shape = tuple(len(axis) for axis in axes)
    if v_euclidean.shape != expected_shape or v_dce.shape != expected_shape:
        raise ValueError("Invalid array shapes.")

    if not np.isfinite(v_euclidean).all() or not np.isfinite(v_dce).all():
        raise ValueError("Non-finite values detected.")

    # Normalized distance to the nearest boundary along each nonperiodic axis.
    margins = {}
    for dim, axis in enumerate(axes):
        if dim == 2:  # Periodic relative heading.
            continue
        width = float(axis[-1] - axis[0])
        if width <= 0:
            raise ValueError(f"Invalid axis: {STATE_NAMES[dim]}")
        margins[dim] = np.minimum(
            axis - axis[0], axis[-1] - axis
        ) / width

    best_distance = -np.inf
    best_difference = -np.inf
    best_index = None

    # Search by slabs to limit memory use.
    for k in range(expected_shape[0]):
        difference = (
            v_dce[k].astype(np.float64)
            - v_euclidean[k].astype(np.float64)
        )
        eligible = difference > 1

        if not eligible.any():
            continue

        distance = np.full(difference.shape, margins[0][k])

        for dim in (1, 3, 4, 5):
            shape = [1] * difference.ndim
            shape[dim - 1] = len(axes[dim])
            np.minimum(
                distance,
                margins[dim].reshape(shape),
                out=distance,
            )

        local_distance = float(distance[eligible].max())
        candidates = eligible & np.isclose(
            distance, local_distance, rtol=0, atol=1e-12
        )
        scores = np.where(candidates, difference, -np.inf)
        local_index = np.unravel_index(np.argmax(scores), scores.shape)
        local_difference = float(difference[local_index])

        tied = np.isclose(
            local_distance, best_distance, rtol=0, atol=1e-12
        )
        if (
            local_distance > best_distance + 1e-12
            or (tied and local_difference > best_difference)
        ):
            best_distance = local_distance
            best_difference = local_difference
            best_index = (k, *local_index)

    if best_index is None:
        print("No grid node satisfies V_DCE > V_Euclidean.")
    else:
        print("Most interior state satisfying V_DCE > V_Euclidean:\n")

        for name, unit, axis, i in zip(
            STATE_NAMES, UNITS, axes, best_index
        ):
            print(f"{name:10s} = {float(axis[i]): .8g} {unit}")

        print(f"\nGrid index: {best_index}")
        print(f"Target time: {float(euclidean['target_time']):g} s")
        print(f"V_Euclidean = {float(v_euclidean[best_index]):.10g}")
        print(f"V_DCE       = {float(v_dce[best_index]):.10g}")
        print(f"Difference  = {best_difference:.10g}")
        print(
            "Minimum relative distance from boundaries: "
            f"{100 * best_distance:.2f}% of the corresponding axis range"
        )

del v_euclidean, v_dce

Most interior state satisfying V_DCE > V_Euclidean:

x_rel      =  3 m
y_rel      =  0 m
theta_rel  = -1.9634955 rad
v_H        =  2.4285715 m/s
delta_E    = -0.15707964 rad
v_E        =  2.4285715 m/s

Grid index: (15, np.int64(8), np.int64(9), np.int64(1), np.int64(2), np.int64(1))
Target time: -3 s
V_Euclidean = -4.990045071
V_DCE       = -3.919091463
Difference  = 1.070953608
Minimum relative distance from boundaries: 14.29% of the corresponding axis range


# Verifica valore brt dato uno stato

In [13]:
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from IPython.display import display


DATA_DIRECTORY = Path("../results/brt")

# Select the metrics to evaluate.
METRICS = (
    "euclidean",
    "dce",
    "btn",
    "stn",
    "long_lat",
    "ttc",
    "time_eggert",
    "rss",
    "eggert",
)

# Relative state: x_rel, y_rel, theta_rel, v_H, delta_E, v_E.
STATE = np.array([10.0, 4.0, 0.0, 6.0, 0.0, 6.0])

STATE_NAMES = (
    "x_rel", "y_rel", "theta_rel", "v_H", "delta_E", "v_E",
)
CORNERS = np.array(list(product((0, 1), repeat=6)), dtype=int)


def interpolation_weights(axes, state, periodic_dims, grid_lo, grid_hi):
    lower, upper, weights = [], [], []

    for dim, axis in enumerate(axes):
        value = float(state[dim])

        if dim in periodic_dims:
            period = float(grid_hi[dim] - grid_lo[dim])
            value = (value - axis[0]) % period + axis[0]
        elif not axis[0] <= value <= axis[-1]:
            raise ValueError(
                f"{STATE_NAMES[dim]}={value:g} outside "
                f"[{axis[0]:g}, {axis[-1]:g}]."
            )

        i = int(np.clip(
            np.searchsorted(axis, value, side="right") - 1,
            0, len(axis) - 1,
        ))

        if dim in periodic_dims and i == len(axis) - 1:
            j = 0
            right = axis[0] + period
        else:
            i = min(i, len(axis) - 2)
            j = i + 1
            right = axis[j]

        lower.append(i)
        upper.append(j)
        weights.append((value - axis[i]) / (right - axis[i]))

    indices = np.where(CORNERS, upper, lower)
    weights = np.asarray(weights)
    corner_weights = np.prod(
        np.where(CORNERS, weights, 1 - weights), axis=1
    )

    return tuple(indices[:, dim] for dim in range(6)), corner_weights


if STATE.shape != (6,) or not np.isfinite(STATE).all():
    raise ValueError("STATE must contain six finite values.")

rows = []

for metric in METRICS:
    path = DATA_DIRECTORY / f"{metric}.npz"

    with np.load(path, allow_pickle=False) as data:
        axes = tuple(np.asarray(data[name], dtype=float)
                     for name in STATE_NAMES)

        if any(
            axis.ndim != 1 or len(axis) < 2
            or not np.isfinite(axis).all()
            or not np.all(np.diff(axis) > 0)
            for axis in axes
        ):
            raise ValueError(f"{metric}: invalid grid axes.")

        indices, weights = interpolation_weights(
            axes, STATE,
            tuple(int(i) for i in data["periodic_dims"]),
            data["grid_lo"], data["grid_hi"],
        )

        values = {}
        for field in ("V0", "BRT"):
            array = data[field]
            if array.shape != tuple(len(axis) for axis in axes):
                raise ValueError(f"{metric}/{field}: incompatible shape.")

            corners = array[indices]
            if not np.isfinite(corners).all():
                raise ValueError(f"{metric}/{field}: non-finite values.")

            values[field] = float(weights @ corners)
            del array

        rows.append({
            "Metric": metric,
            "Time [s]": float(data["target_time"]),
            "BRT": values["BRT"],
            "BRT classification": (
                "Safe" if values["BRT"] > 0
                else "Unsafe" if values["BRT"] < 0
                else "Boundary"
            ),
            "V0": values["V0"],
            "Inside terminal set": values["V0"] <= 0,
        })

results = pd.DataFrame(rows)

print("Relative state:", dict(zip(STATE_NAMES, STATE)))
display(results.style.format({
    "Time [s]": "{:.2f}",
    "BRT": "{:.6f}",
    "V0": "{:.6f}",
}).hide(axis="index"))

if (results["BRT"] > 0).all():
    print("BRT is positive for all selected metrics.")
elif (results["BRT"] < 0).all():
    print("BRT is negative for all selected metrics.")
else:
    print("The selected metrics do not all assign the same strict sign.")

Relative state: {'x_rel': np.float64(10.0), 'y_rel': np.float64(4.0), 'theta_rel': np.float64(0.0), 'v_H': np.float64(6.0), 'delta_E': np.float64(0.0), 'v_E': np.float64(6.0)}


Metric,Time [s],BRT,BRT classification,V0,Inside terminal set
euclidean,-3.00,5.506484,Safe,5.872983,False
dce,-3.00,4.802478,Safe,5.543852,False
btn,-3.00,5.505924,Safe,5.836954,False
stn,-3.00,4.883365,Safe,5.598368,False
long_lat,-3.00,4.892438,Safe,5.598368,False
ttc,-3.00,2.999110,Safe,3.000000,False
time_eggert,-3.00,2.843292,Safe,3.000000,False
rss,-3.00,0.470577,Safe,0.471405,False
eggert,-3.00,0.799630,Safe,0.799955,False


BRT is positive for all selected metrics.
